In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def readImages(folder, num_images):
    arr_images = []
    for i in range(num_images):
        arr_images.append(cv2.imread(f'{folder}{i}.jpg'))
    return np.array(arr_images, dtype=np.float32)

def images_to_gray(images: np.array):
    result = []
    
    for i in range(len(images)):
        result.append(cv2.cvtColor(images[i], cv2.COLOR_BGR2GRAY))
    
    return result

folder = '../images/'
im = readImages(folder, 2)

In [ ]:
def project_images_onto_plane(img1, img2):


    # Detect keypoints
    sift = cv2.SIFT_create()
    keypoints1, descriptors1 = sift.detectAndCompute(img1, None)
    keypoints2, descriptors2 = sift.detectAndCompute(img2, None)
    
    
    # Match Descriptors
    index_params = dict(algorithm=1, trees=5)
    search_params = dict(checks=50)
    matcher = cv2.FlannBasedMatcher(index_params, search_params)
    matches = matcher.knnMatch(descriptors1, descriptors2, k=2)
    
    good_matches = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)
            
            
    if len(good_matches) > 4:
        src_pts = []
        tgt_pts = []
        
        for match in good_matches:
            kp1 = keypoints1[match.queryIdx].pt
            src_pts.append(kp1)
            
            kp2 = keypoints2[match.trainIdx].pt
            tgt_pts.append(kp2)
            
        src_pts = np.float32(src_pts)
        tgt_pts = np.float32(tgt_pts)
        
        threshold = 5
        H, inliers = cv2.findHomography(
            src_pts,
            tgt_pts,
            cv2.RANSAC,
            threshold
        )
        
        height, width = img2.shape[:2]
        
        warped = cv2.warpPerspective(
            img1,
            H,
            (width, height)
        )
        
        return warped
    else:
        print("Not enough matches are found - %d/%d" % (len(good_matches), 5))
        return None

folder = '../images/'
images = readImages(folder, 2)
images_gray = images_to_gray(images)

fig, ax = plt.subplots(figsize=(10, 10))

# print(images_gray[0])

warped = project_images_onto_plane(images_gray[0], images_gray[1])
ax.imshow(warped)
plt.show()